In [ ]:
%pip install plotly
%pip install --upgrade nbformat
%pip install dash


[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
  Using cached flask-3.0.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached MarkupSafe-3.0.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (4.0 kB)
  Using cached charset_normalizer-3.4.1-cp313-cp313-macosx_10_13_universal2.whl.metadata (35 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached certifi-2025.1.31-py3-none-any.whl.metadata (2.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 3.8 MB/s eta 0:00:00a 0:00:0

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import textwrap

# ---------------------------
# 1) Read and parse the CSV
# ---------------------------
file_path = '/Users/junjie/lab10spark/TopEngagementResults.csv'  # Adjust to your actual file path

records = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # Locate the first and last commas
        first_comma = line.find(',')
        last_comma = line.rfind(',')
        
        # Skip if the format is not as expected
        if first_comma == -1 or last_comma == -1 or first_comma == last_comma:
            continue
        
        # Extract fields: category, product_title, rating
        category = line[:first_comma]
        product_title = line[first_comma + 1 : last_comma]
        
        try:
            rating = int(line[last_comma + 1 :])
        except ValueError:
            # Skip if rating is not an integer
            continue
        
        records.append({
            'category': category,
            'product_title': product_title,
            'rating': rating
        })

# ---------------------------
# 2) Create a DataFrame
# ---------------------------
df = pd.DataFrame(records)

# ---------------------------
# 3) Define a function to wrap labels
# ---------------------------
def wrap_labels(label, width=30):
    return "\n".join(textwrap.wrap(label, width))

# -----------------------------------------------------
# 4) Create one bar trace per category (top 5 products)
# -----------------------------------------------------
categories = df['category'].unique()
traces = []

# Calculate the maximum rating for the x-axis range across all categories
for cat in categories:
    group_data = df[df['category'] == cat].copy()
    # Sort by rating descending and select top 5
    top5 = group_data.sort_values(by='rating', ascending=False).head(5)
    
    if top5.empty:
        continue
    
    # Wrap long product titles
    top5['wrapped_title'] = top5['product_title'].apply(lambda x: wrap_labels(x, 30))
    
    # Create a Bar trace for the category (initially hidden)
    trace = go.Bar(
        y=top5['wrapped_title'],
        x=top5['rating'],
        name=cat,
        orientation='h',
        visible=False
    )
    traces.append(trace)

# Make the first category’s trace visible by default
if traces:
    traces[0].visible = True

# ------------------------------------------------
# 5) Build the update menus (dropdown) for Plotly
# ------------------------------------------------
buttons = []
for i, cat in enumerate(categories):
    buttons.append({
        'label': cat,
        'method': 'update',
        'args': [
            {'visible': [j == i for j in range(len(traces))]},  # show only the i-th trace
            {
                'title': f'Top 5 Most User-Engaged Products for Category: {cat}',
                'xaxis': {
                    'range': [0, df[df['category'] == cat]['rating'].max() * 1.1]  # dynamic x-axis range for each category
                }
            }
        ]
    })

# ----------------------------------------
# 6) Create the figure with the layout
# ----------------------------------------
layout = go.Layout(
    title=f"Top 5 Most User-Engaged Products for Category: {categories[0] if categories.size else 'N/A'}",
    updatemenus=[{
        'buttons': buttons,
        'direction': 'down',
        'showactive': True,
        'active': 0,
        'x': 0.1,
        'xanchor': 'left',
        'y': 1.15,
        'yanchor': 'top'
    }],
    xaxis={
        'title': 'Number of Ratings',
        'rangemode': 'tozero',   # Ensure the x-axis starts at 0
    },
    yaxis={'title': 'Product Titles'},
    height=700,
    width=1200,
    margin=dict(l=150, r=50, t=50, b=100),
)

fig = go.Figure(data=traces, layout=layout)
fig.show()


In [15]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ----------------------------
# 1) Load and Prepare the Data
# ----------------------------
df = pd.read_csv("rolling_time_window_with_category.csv", parse_dates=["timestamp"])

# Aggregate data by category and timestamp
cat_monthly = df.groupby(["new_category", "timestamp"])["rolling_avg_rating"].mean().reset_index()

# Forecast 24 months (2 years) beyond the last available timestamp
# Adjust as needed; your original code mentioned 480 for 40 years
future_periods = 240

# Get the unique categories
categories = cat_monthly["new_category"].unique()

# This will hold all the Plotly traces
all_traces = []

# We'll also keep track of which categories are valid (not skipped)
valid_categories = []

# -------------------------------------
# 2) Build Historical + Forecast Traces
# -------------------------------------
for cat in categories:
    # Filter data for this category
    data = cat_monthly[cat_monthly["new_category"] == cat].sort_values("timestamp").copy()
    
    # Basic validation checks
    if len(data) < 2 or data["rolling_avg_rating"].isna().all():
        print(f"Skipping forecast for '{cat}' - insufficient or invalid data.")
        continue
    
    # Convert timestamp to numeric
    data["time_numeric"] = data["timestamp"].map(pd.Timestamp.toordinal)
    
    X = data["time_numeric"].values
    y = data["rolling_avg_rating"].values
    
    # Check if we have at least two distinct timestamps and ratings
    if len(np.unique(X)) < 2:
        print(f"Skipping forecast for '{cat}' - all timestamps are the same.")
        continue
    if len(np.unique(y[~np.isnan(y)])) < 2:
        print(f"Skipping forecast for '{cat}' - ratings are all the same or NaN.")
        continue
    
    # Fit a linear model (degree=1 polynomial)
    try:
        coef = np.polyfit(X, y, 1)  # slope, intercept
    except Exception:
        print(f"Skipping forecast for '{cat}' - error in polyfit (possibly poorly conditioned).")
        continue
    
    poly1d_fn = np.poly1d(coef)
    
    # Forecast future months
    last_date = data["timestamp"].max()
    future_dates = [last_date + pd.DateOffset(months=i) for i in range(1, future_periods + 1)]
    future_numeric = np.array([d.toordinal() for d in future_dates])
    future_preds = poly1d_fn(future_numeric)
    
    # If the forecast is NaN, skip
    if np.isnan(future_preds).any():
        print(f"Skipping forecast for '{cat}' - forecast produced NaN values.")
        continue
    
    # 2.1) Create the Historical Trace
    hist_trace = go.Scatter(
        x = data["timestamp"],
        y = data["rolling_avg_rating"],
        mode = "lines+markers",
        name = f"{cat} - Historical",
        visible = False  # We'll toggle visibility via dropdown
    )
    
    # 2.2) Create the Forecast Trace
    forecast_trace = go.Scatter(
        x = future_dates,
        y = future_preds,
        mode = "lines+markers",
        line = dict(dash="dash"),
        name = f"{cat} - Forecast",
        visible = False
    )
    
    # Append both traces
    all_traces.append(hist_trace)
    all_traces.append(forecast_trace)
    
    valid_categories.append(cat)

# If no valid categories/traces, just stop
if not valid_categories:
    print("No valid categories to plot.")
else:
    # -------------------------------------------------
    # 3) Create the Dropdown Buttons for Each Category
    # -------------------------------------------------
    buttons = []
    
    # Each category corresponds to two traces in the list (historical + forecast)
    # So for category i, the traces are at indices 2*i and 2*i + 1
    for i, cat in enumerate(valid_categories):
        # Generate a "visibility mask" for all traces
        # We only want to make the i-th category's 2 traces visible
        visibility = [False] * len(all_traces)
        visibility[2*i] = True     # Historical
        visibility[2*i + 1] = True # Forecast
        
        buttons.append({
            "label": cat,
            "method": "update",
            "args": [
                {"visible": visibility},
                {"title": f"Historical and Forecasted Rolling Average Rating<br>Category: {cat}"}
            ]
        })
    
    # Make the first category's traces visible by default
    if len(valid_categories) > 0:
        all_traces[0].visible = True
        all_traces[1].visible = True
    
    # -------------------------------------------
    # 4) Construct the Figure and Add the Layout
    # -------------------------------------------
    fig = go.Figure(data=all_traces)
    
    # Add the dropdown menu
    fig.update_layout(
        title = f"Historical and Forecasted Rolling Average Rating<br>Category: {valid_categories[0]}",
        xaxis_title = "Time",
        yaxis_title = "Rolling Average Rating",
        updatemenus = [
            {
                "buttons": buttons,
                "direction": "down",
                "showactive": True,
                "active": 0,
                "x": 1.05,
                "xanchor": "right",
                "y": 1.15,
                "yanchor": "top"
            }
        ],
        # Figure size/margins
        height=600,
        width=1000,
        margin=dict(l=80, r=40, t=80, b=80),
    )
    
    # Improve x-axis formatting (e.g., for dates)
    fig.update_xaxes(rangemode="normal")
    
    # Display the figure
    fig.show()


Skipping forecast for 'Cell Phone & Camera w. Accessories' - forecast produced NaN values.
Skipping forecast for 'Daily Gadgets' - forecast produced NaN values.
Skipping forecast for 'Media' - forecast produced NaN values.
Skipping forecast for 'Sports & Health' - forecast produced NaN values.
Skipping forecast for 'Toys & Games' - forecast produced NaN values.
